

# M3 3 Gen AI ETL
### OPIM 5512 — Applied Data Science · Module3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module3/M3_3_GenAI_ETL.ipynb)

*Run me top to bottom — **Runtime → Run all**. Data loads from a stable link, so there's nothing to upload.*

# Using GenAI to do your ETL
---------------------
**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**


Reference on Prompt Engineering with LLMs (like Gemini):
* [Prompt Engineering: Overview and Guide from Google](https://cloud.google.com/discover/what-is-prompt-engineering?hl=en)

Text data is gold - it used to be too big or complicated to work with ten years ago. That's changed since 2022... let's show you how to use GenAI to enhance in a more flexible way. Set the constraints for your LLM call and rock it! No RegEx here - just LLMs doing what they do best (sometimes!)

This AWESOME LLM function doesn’t talk to Craigslist at all. It assumes two earlier steps already happened:
* Scraper wrote .txt files for each listing.
* Classic ETL wrote per-listing JSONL files with regex-based fields.

This extractor:
* Reads those existing JSONL records (structured/run_id=…/jsonl/*.jsonl).
* Follows source_txt back to the raw .txt.
* Calls Gemini to re-extract the same key fields.
* Writes a sibling file into jsonl_llm/ with _llm in the filename.

We’re not replacing our ETL; we’re adding an LLM layer on top and saving the results side-by-side.

# Important Aspects of our LLM-powered ETL

## 🔹 Clean separation via environment variables
Up top:

```python
PROJECT_ID        = os.getenv("PROJECT_ID", "")
REGION            = os.getenv("REGION", "us-central1")
BUCKET_NAME       = os.getenv("GCS_BUCKET", "")
STRUCTURED_PREFIX = os.getenv("STRUCTURED_PREFIX", "structured")
LLM_PROVIDER      = os.getenv("LLM_PROVIDER", "vertex").lower()
LLM_MODEL         = os.getenv("LLM_MODEL", "gemini-2.5-flash")
```

* Model, region, bucket, prefixes are config, not hard-coded.
* You can switch models or buckets by changing env vars, without touching code.

## 🔹Strict JSON schema + `response_schema`

In `_vertex_extract_fields`:

```python
schema = {
    "type": "object",
    "properties": {
        "price":   {"type": "integer", "nullable": True},
        "year":    {"type": "integer", "nullable": True},
        "make":    {"type": "string",  "nullable": True},
        "model":   {"type": "string",  "nullable": True},
        "mileage": {"type": "integer", "nullable": True},
    },
    "required": ["price", "year", "make", "model", "mileage"]
}
```

And then:

```python
gen_cfg = GenerationConfig(
    temperature=0.0,
    response_mime_type="application/json",
    response_schema=schema,
)
```

* We're using Gemini’s structured outputs `(schema)` instead of “just give me JSON please”.
* `nullable`: True sets a clear contract: “If missing, use null.”
* `Temperature 0.0` reinforces “this is extraction, not creativity.”

## 🔹 Prompt-design/System-style rules embedded in the prompt

```python
sys_instr = (
    "Extract ONLY the following fields from the input text. "
    "Return a strict JSON object that conforms to the provided schema. "
    "If a value is not present, use null. "
    "Rules: integers for price/year/mileage; price in USD; mileage in miles; "
    "do not infer values not explicitly present; do not add extra keys."
)

prompt = f"{sys_instr}\n\nTEXT:\n{raw_text}"
```

* Clear constraints: no hallucinating, no extra keys, units specified.
* This is a concrete example of “don’t invent values if you don’t see them.”
* Shows the pattern of combining “instructions” + “data” into a single prompt string.

## 🔹 Robust retry logic for LLM calls

They define:

```python
def _if_llm_retryable(exception):
    return isinstance(exception, (ResourceExhausted, InternalServerError, Aborted, DeadlineExceeded))
```

Then:
```python
max_attempts = 3
for attempt in range(max_attempts):
    try:
        resp = model.generate_content(prompt, generation_config=gen_cfg)
        break
    except Exception as e:
        if not _if_llm_retryable(e) or attempt == max_attempts - 1:
            raise
        sleep_time = LLM_RETRY._calculate_sleep(attempt)
        time.sleep(sleep_time)
```

This is gold for “production-ish” patterns:
* Don’t treat one flaky LLM call as a crash of the whole job.
* Retry only on transient exceptions (quota, internal, aborted).
* Sleeps/backs off for politeness.

## 🔹 Post-processing & normalization (`_safe_int`, `_norm_str`)
After parsing:

```python
def _safe_int(x):
    try:
        if x is None or x == "":
            return None
        return int(str(x).replace(",", "").strip())
    except Exception:
        return None

parsed["price"]   = _safe_int(parsed.get("price"))
parsed["year"]    = _safe_int(parsed.get("year"))
parsed["mileage"] = _safe_int(parsed.get("mileage"))
...
def _norm_str(s):
    if s is None: return None
    s = str(s).strip()
    return s if s else None

parsed["make"]  = _norm_str(parsed.get("make"))
parsed["model"] = _norm_str(parsed.get("model"))
```

* Even when the LLM returns JSON, you don’t fully trust the types.
* You still coerce and clean on your side.

**On Your Own:** What additional normalization would you add? Lowercasing makes? Mapping ‘VW’ → ‘Volkswagen’?

## 🔹 Separation of "classic" ETL vs "LLM" outputs in storage

Spot this line:
```python
out_prefix = in_key.rsplit("/", 2)[0] + "/jsonl_llm"
out_key    = out_prefix + f"/{post_id}_llm.jsonl"
```

So if classic ETL produced:
* structured/run_id=20251026170002/jsonl/1234567890.jsonl

then the LLM output goes to:
*structured/run_id=20251026170002/jsonl_llm/1234567890_llm.jsonl

For now, we will not overwrite your classic ETL; write LLM-enriched data to a parallel, clearly named location so you can compare them. But be aware, if you kill the traditional ETL job, THIS job will fail... because it won't find the original ETL data to extract the path from! 💡

## 🔹 Tracking who and when did the LLM work
The output record includes:
* "llm_provider": "vertex",
* "llm_model":    LLM_MODEL,
* "llm_ts":       datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),


These three fields let you answer later:
* Which model produced these fields?
* When was the extraction run?
* Could you re-run with a newer model and compare?

## 🔹Overwrite logic

In the HTTP handler:
```python
if not overwrite and _blob_exists(out_key):
    skipped += 1
    continue
```

Together with `OVERWRITE_DEFAULT` env var (see top of the `main.py` script for LLM ETL) and overwrite in the request body, this gives:
* Ability to safely re-run on the same run_id without duplicating work.
* Controllable behavior: default “don’t overwrite” unless you explicitly say so.

This is a nice 'oopsy' preventer - what happens if the function crashes halfway though? How do you restart without redoing everything?

# On Your Own

## Adding More Fields to the Schema
Add new fields in the schema inside `_vertex_extract_fields()`

Here’s the schema section:
```python
schema = {
    "type": "object",
    "properties": {
        "price":   {"type": "integer", "nullable": True},
        "year":    {"type": "integer", "nullable": True},
        "make":    {"type": "string",  "nullable": True},
        "model":   {"type": "string",  "nullable": True},
        "mileage": {"type": "integer", "nullable": True},
    },
    "required": ["price", "year", "make", "model", "mileage"]
}
```

If you want the LLM to extract anything else, like:
* transmission
* body_type
* fuel
* color
* title_status
* condition
* location (City, State, zip code)
* Other things you think of!

You must add it to the schema.

Example:
```python
"transmission": {"type": "string", "nullable": True},"
```

and optionally add it to "required" (but sometimes, you don’t want these required due to hallucination).

### Test new LLM-PoC on a specific folder and overwrite

```python
davesdatascience@cloudshell:~ (myscrapers-dww05002-v2)$ curl -X POST "https://extractor-llm-poc-768638756832.us-central1.run.app" \
  -H "Authorization: bearer $(gcloud auth print-identity-token)" \
  -H "Content-Type: application/json" \
  -d '{
        "name": "Developer",
        "overwrite": true,
        "run_id": "20251208160006"
      }'
```

You should get an output like this:
```python
{"errors":0,"ok":true,"processed":50,"run_id":"20251208160006","skipped":0,"version":"extractor-llm-poc","written":50}
```

## Write A New `materialize-llm` function and deploy to GCP!
You should be able to hack your existing materialize cloud function... and make a copy of it called `materialize-llm` and start aggregating all of the new LLM data you are creating. You can have the job run once and hour or once a day. The most important thing is that you have growing record of LLM outputs.

All you really need to do is change the path it reads from and the path/file name it writes to. Simply add a `-llm` suffix to the end of everything.

However, if you DON'T make a new Cloud Run function, it will stop materializing all of your RegEx-generated/traditional ETL data. Personally, for class work, I think it's more exciting to leave both and THEN build model to see which one does better. You can either build ML models or some type of dashboard that shows "average car price per day" or number of "Toyotas for sale per day" or anything else that you think could be useful to an end user.

## Get rid of the reliance on the other/old jsonl path
It seems logical that one day, you MAY get rid of the traditional ETL. If you do, then your LLM-ETL will break because the path doesn't exist. Update the LLM-ETL code so that it doesn't need some random path in order to make a new path - make it directly from the raw .txt file name.